# 00 — GCP Setup and Data Validation

Run this notebook first on Vertex AI Workbench / Colab Enterprise before any training.

**Steps covered:**
1. Authenticate with GCP and verify bucket access
2. Validate that all 1014 PDFs are present in the bucket
3. Run `clean_labels.py` if CSVs are missing from the bucket
4. Run `render_pdf.render_all()` to populate `rendered_pages/`
5. Verify rendered images were written back to GCS

> **GCSFuse:** On Vertex AI Workbench, your GCS bucket is automatically mounted at `/gcs/<bucket_name>/`.  
> All code in `src/` reads and writes via these local-looking paths — no code changes required.

## 0. Configuration

**Set your bucket name here.** Update `configs/*.yaml` with the same value.

In [ ]:
BUCKET_NAME = "YOUR_BUCKET_NAME"  # <-- replace with your GCS bucket name
PROJECT_ROOT = "/home/jupyter/for_tal"  # adjust if your repo is cloned elsewhere

GCS_MOUNT = f"/gcs/{BUCKET_NAME}"

# Paths via GCSFuse mount
RAW_PDF_DIR  = f"{GCS_MOUNT}/raw_pdfs"
RENDERED_DIR = f"{GCS_MOUNT}/rendered_pages"
DATA_DIR     = f"{GCS_MOUNT}/data"
METADATA_CSV = f"{DATA_DIR}/metadata.csv"

print(f"Bucket: gs://{BUCKET_NAME}")
print(f"GCSFuse mount: {GCS_MOUNT}")

## 1. Authentication

On Vertex AI Workbench the VM service account is pre-authenticated.  
On Colab Enterprise run the cell below.

In [ ]:
import sys

# Colab Enterprise auth — skip on Vertex AI Workbench (already authenticated)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab auth complete")
except ImportError:
    print("Not running in Colab — using VM service account credentials (Vertex AI Workbench)")

# Verify credentials work
from google.cloud import storage
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
bucket.reload()
print(f"Connected to gs://{BUCKET_NAME}")

## 2. Verify GCSFuse Mount

In [ ]:
import os
from pathlib import Path

mount_path = Path(GCS_MOUNT)
if not mount_path.exists():
    print(f"ERROR: GCSFuse mount not found at {GCS_MOUNT}")
    print("On Vertex AI Workbench, ensure 'Mount GCS bucket' is enabled in instance settings.")
    print("Alternatively, run: gcsfuse --implicit-dirs YOUR_BUCKET_NAME /gcs/YOUR_BUCKET_NAME")
else:
    contents = list(mount_path.iterdir())
    print(f"GCSFuse mount OK: {GCS_MOUNT}")
    print(f"Top-level entries: {[p.name for p in contents]}")

## 3. Validate PDF Count in Bucket

In [ ]:
import pandas as pd

EXPECTED_PDF_COUNT = 1014

raw_pdf_path = Path(RAW_PDF_DIR)

if not raw_pdf_path.exists():
    print(f"raw_pdfs/ directory not found at {RAW_PDF_DIR}")
    print("Run scripts/drive_to_gcs.py to upload PDFs first.")
    print(f"  python scripts/drive_to_gcs.py --bucket {BUCKET_NAME}")
else:
    pdf_files = list(raw_pdf_path.glob("*.pdf"))
    print(f"PDFs in bucket: {len(pdf_files)} / {EXPECTED_PDF_COUNT} expected")
    if len(pdf_files) < EXPECTED_PDF_COUNT:
        print(f"WARNING: {EXPECTED_PDF_COUNT - len(pdf_files)} PDFs missing.")
        print("Re-run drive_to_gcs.py (--no-skip-existing to force re-upload).")
    elif len(pdf_files) == EXPECTED_PDF_COUNT:
        print("PDF count OK")

## 4. Validate Metadata CSVs

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

metadata_path = Path(METADATA_CSV)

if not metadata_path.exists():
    print(f"metadata.csv not found at {METADATA_CSV}")
    print("Run drive_to_gcs.py (which uploads CSVs) or manually copy from local data/ directory.")
else:
    df = pd.read_csv(metadata_path)
    print(f"metadata.csv shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nLabel distribution:")
    print(df['label_binary'].value_counts().rename({0: 'safe (0)', 1: 'risky (1)'}))
    print(f"\nInstitution distribution:")
    print(df['institution'].value_counts())
    print(f"\nRows with split assigned: {(df['split'] != '').sum()}")
    
    # Check for rubric annotation
    rubric_done = (df['D'] != -1).sum()
    print(f"\nRubric annotated rows: {rubric_done} / {len(df)}")
    if rubric_done == 0:
        print("NOTE: D/H/S/L rubric scores not yet populated. Run annotation before DiT training.")

## 5. Generate Splits (if not already done)

In [ ]:
import yaml

# Load config to get split settings
with open(f"{PROJECT_ROOT}/configs/baseline.yaml") as f:
    cfg = yaml.safe_load(f)

splits_dir = Path(f"{DATA_DIR}/splits")
expected_splits = ["train.csv", "val.csv", "test.csv"]
splits_present = all((splits_dir / s).exists() for s in expected_splits)

if splits_present:
    print("Splits already exist:")
    for s in expected_splits:
        split_df = pd.read_csv(splits_dir / s)
        print(f"  {s}: {len(split_df)} rows")
else:
    print("Splits not found — generating now...")
    from src.data.splits import create_grouped_splits, save_splits
    
    metadata_df = pd.read_csv(METADATA_CSV)
    group_col = cfg['splits']['group_col']  # template_family
    ratios = {
        'train': cfg['splits']['train_ratio'],
        'val': cfg['splits']['val_ratio'],
        'test': cfg['splits']['test_ratio'],
    }
    metadata_df = create_grouped_splits(metadata_df, group_col=group_col, ratios=ratios)
    save_splits(metadata_df, output_dir=str(splits_dir))
    
    # Save updated metadata with split column back to GCS
    metadata_df.to_csv(METADATA_CSV, index=False)
    print("Splits generated and metadata.csv updated.")
    for s in expected_splits:
        sdf = pd.read_csv(splits_dir / s)
        print(f"  {s}: {len(sdf)} rows")

## 5b. Validate Split Label Mix

Checks that each split (train / val / test) contains **both** label classes and that
no split is dominated by a single class.

**Why this matters:** `institution` is perfectly correlated with label in the current
dataset (`regular_docs` = safe, `questionnaires` = risky). If `institution` were used
as the grouping key, one entire split could end up with only one class.  
The configs use `template_family` instead, but we verify the result here regardless.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

splits_dir = Path(f"{DATA_DIR}/splits")
split_names = ["train", "val", "test"]
EXPECTED_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
MIN_MINORITY_FRACTION = 0.10   # flag if any class is below 10% in a split

all_ok = True
split_stats = []

print("=" * 60)
print(f"{'Split':<8} {'N':>6}  {'Safe (0)':>10}  {'Risky (1)':>10}  {'Risky%':>8}  {'Status'}")
print("=" * 60)

for name in split_names:
    csv_path = splits_dir / f"{name}.csv"
    if not csv_path.exists():
        print(f"{name:<8}  NOT FOUND — run cell 5 first")
        all_ok = False
        continue

    sdf = pd.read_csv(csv_path)
    n = len(sdf)
    n_safe  = (sdf["label_binary"] == 0).sum()
    n_risky = (sdf["label_binary"] == 1).sum()
    risky_pct = n_risky / n if n > 0 else 0

    issues = []
    if n_safe == 0:
        issues.append("NO SAFE SAMPLES")
    if n_risky == 0:
        issues.append("NO RISKY SAMPLES")
    if risky_pct < MIN_MINORITY_FRACTION:
        issues.append(f"risky < {MIN_MINORITY_FRACTION:.0%}")
    if risky_pct > (1 - MIN_MINORITY_FRACTION):
        issues.append(f"safe < {MIN_MINORITY_FRACTION:.0%}")

    status = "OK" if not issues else "WARNING: " + ", ".join(issues)
    if issues:
        all_ok = False

    print(f"{name:<8} {n:>6}  {n_safe:>10}  {n_risky:>10}  {risky_pct:>7.1%}  {status}")
    split_stats.append({"name": name, "n": n, "safe": n_safe, "risky": n_risky, "risky_pct": risky_pct})

print("=" * 60)

if all_ok:
    print("\nSplit mix: PASS — all splits contain both classes above threshold.")
else:
    print("\nSplit mix: FAIL — review warnings above before training.")
    print("Consider switching configs/baseline.yaml splits.strategy to 'kfold'")
    print("or manually adjusting the group_col to better balance classes.")

# --- Institution breakdown per split ---
print("\nInstitution distribution per split:")
metadata_df = pd.read_csv(METADATA_CSV)
if "split" in metadata_df.columns and metadata_df["split"].notna().any():
    cross = pd.crosstab(metadata_df["split"], metadata_df["institution"])
    print(cross.to_string())
else:
    print("  'split' column not yet populated in metadata.csv — run cell 5 first.")

# --- Bar chart ---
if split_stats:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Absolute counts
    ax = axes[0]
    x = np.arange(len(split_stats))
    width = 0.35
    safe_vals  = [s["safe"]  for s in split_stats]
    risky_vals = [s["risky"] for s in split_stats]
    ax.bar(x - width/2, safe_vals,  width, label="Safe (0)",  color="#4C9BE8")
    ax.bar(x + width/2, risky_vals, width, label="Risky (1)", color="#E8704C")
    ax.set_xticks(x)
    ax.set_xticklabels([s["name"] for s in split_stats])
    ax.set_ylabel("Sample count")
    ax.set_title("Label counts per split")
    ax.legend()
    for i, (sv, rv) in enumerate(zip(safe_vals, risky_vals)):
        ax.text(i - width/2, sv + 2, str(sv), ha="center", fontsize=8)
        ax.text(i + width/2, rv + 2, str(rv), ha="center", fontsize=8)

    # Risky fraction
    ax2 = axes[1]
    risky_pcts = [s["risky_pct"] * 100 for s in split_stats]
    bars = ax2.bar([s["name"] for s in split_stats], risky_pcts, color=["#4C9BE8", "#A8D8A8", "#E8704C"])
    ax2.axhline(MIN_MINORITY_FRACTION * 100, color="red", linestyle="--", linewidth=1, label=f"Min threshold ({MIN_MINORITY_FRACTION:.0%})")
    ax2.axhline((1 - MIN_MINORITY_FRACTION) * 100, color="red", linestyle="--", linewidth=1)
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("Risky % in split")
    ax2.set_title("Class balance per split")
    ax2.legend()
    for bar, pct in zip(bars, risky_pcts):
        ax2.text(bar.get_x() + bar.get_width()/2, pct + 1, f"{pct:.1f}%", ha="center", fontsize=9)

    plt.suptitle("Split Label Mix Validation", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Render PDFs to Images

Renders all 1014 PDFs to grayscale PNGs at 150 DPI.  
Output is written to `gs://<bucket>/rendered_pages/` via GCSFuse.  
Re-running is safe — existing images are skipped.

In [ ]:
from src.data.render_pdf import render_all

results = render_all(
    pdf_dir=RAW_PDF_DIR,
    output_dir=RENDERED_DIR,
    dpi=150,
    grayscale=True,
    target_size=(224, 224),
    skip_existing=True,
)

status_counts = {}
for r in results:
    s = r['status']
    status_counts[s] = status_counts.get(s, 0) + 1

print(f"Rendering complete: {status_counts}")

# Verify rendered count
rendered_files = list(Path(RENDERED_DIR).glob("*.png"))
print(f"Rendered images in GCS: {len(rendered_files)}")

## 7. Spot-Check Rendered Images

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

rendered_files = list(Path(RENDERED_DIR).glob("*.png"))
sample = random.sample(rendered_files, min(6, len(rendered_files)))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, png_path in zip(axes.flat, sample):
    img = Image.open(png_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(Path(png_path).stem[:40], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample Rendered Pages (150 DPI, 224x224, Grayscale)')
plt.tight_layout()
plt.show()

## 8. Setup Complete — Next Steps

| Step | Notebook / Script |
|------|-------------------|
| Data audit | `01_data_audit.ipynb` |
| Rendering quality check | `02_rendering_checks.ipynb` |
| Label consistency | `03_label_consistency.ipynb` |
| Baseline training (ResNet/ViT) | `04_baseline_training.ipynb` |
| DiT fine-tuning | `05_dit_training.ipynb` |
| Calibration & threshold selection | `06_calibration_eval.ipynb` |

**Before training, complete:**
- [ ] Populate D/H/S/L rubric scores in `metadata.csv` (at least borderline samples)
- [ ] Verify splits do not leak institution-label correlation (see warning in `clean_labels.py`)

**Bucket structure at this point:**
```
gs://YOUR_BUCKET_NAME/
  raw_pdfs/          # 1014 PDFs
  rendered_pages/    # 1014 PNGs (224x224 grayscale)
  data/
    metadata.csv     # master index
    labels_binary_clean.csv
    splits/
      train.csv
      val.csv
      test.csv
```